# NOTE:
# This notebook was originally executed from:
# /content/drive/MyDrive/notebooks/
## Paths may need to be adjusted if executed from a different directory.

This code analyzes and processes data downloaded from ERA5LAND. The data has been collected so far from NASA Power (temperature, relative and specific humidity, and precipitation) and ERA5LAND (total cloud cover).

Information about the data collected from NASA Power:

https://power.larc.nasa.gov/docs/methodology/meteorology/

Information about the data collected from ERA5:

https://www.ecmwf.int/en/forecasts/dataset/ecmwf-reanalysis-v5#:~:text=ERA5%20es%20el%20rean%C3%A1lisis%20atmosf%C3%A9rico,model:%201/a/137

As you can see, NASA Power has a resolution of 50 x 55 km, while ERA5 has a resolution of 31 x 31 km. Although the resolutions appear quite poor and inconsistent, it still provides a good approximation of the data, as the northeastern United States is flat and lacks abrupt topographic changes. Furthermore, there are no weather stations close enough to Delaware to obtain higher-resolution data.

In [ ]:
!pip install cdsapi xarray netcdf4 pandas numpy matplotlib tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 31.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import xarray as xr
import pandas as pd
from tqdm import tqdm

base_path = "/content/drive/MyDrive/Colab Notebooks"

paths = {
    "tcc": os.path.join(base_path, "clouds"),
    "t_h_p_2021": os.path.join(base_path, "cloud_water"),
    "t_h_p_2022": os.path.join(base_path, "conv_prep"),
    "t_h_p_2023": os.path.join(base_path, "large_prep")
}

for name, path in paths.items():
    files = sorted([os.path.join(path, f) for f in os.listdir(path) if f.endswith(".nc")])
    print(f"{name}: {len(files)} files")


tcc: 5 archivos
t_h_p_2021: 5 archivos
t_h_p_2022: 5 archivos
t_h_p_2023: 5 archivos


In [ ]:
import os
import gc
import xarray as xr
import pandas as pd
from tqdm import tqdm

# ============================================
# 1️⃣ Function to load and combine NetCDF files
# ============================================
def load_and_concat(folder_path):

    files = sorted([os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith(".nc")])
    datasets = []

    print(f"Loading {os.path.basename(folder_path)}:")
    for f in tqdm(files):
        ds = xr.open_dataset(f)
        # Remove extra dimensions
        if "valid_time" in ds.dims and "time" not in ds.dims:
            ds = ds.rename({"valid_time": "time"})
        if "expver" in ds.dims:
            ds = ds.isel(expver=0)
        datasets.append(ds)

    ds_combined = xr.concat(datasets, dim="time", join="outer")
    print(f"✅ {folder_path}: Succesfully combined ({len(datasets)} files)")
    return ds_combined


# ============================================
# 2️⃣ Processing functions
# ============================================
def process_tcc(ds):

    ds = ds.rename({"tcc": "cloud_cover"})
    ds["cloud_cover"] = ds["cloud_cover"] * 100
    df = ds[["cloud_cover"]].to_dataframe().reset_index()
    return df


def process_tclw(ds):

    ds = ds.rename({"tclw": "cloud_liquid_water"})
    df = ds[["cloud_liquid_water"]].to_dataframe().reset_index()
    return df


def process_precip(ds, var_name, new_name):

    ds = ds.rename({var_name: new_name})
    ds[new_name] = ds[new_name] * 1000  # Convertir a mm
    df = ds[[new_name]].to_dataframe().reset_index()
    return df


# ============================================
# 3️⃣ Upload the datasets from Google Drive
# ============================================
base_path = "/content/drive/MyDrive/Colab Notebooks"
folders = {
    "tcc": f"{base_path}/clouds",
    "tclw": f"{base_path}/cloud_water",
    "cp": f"{base_path}/conv_prep",
    "lsp": f"{base_path}/large_prep"
}

print("=== Processing ERA5 atmospheric variables ===")

tcc_ds = load_and_concat(folders["tcc"])
tclw_ds = load_and_concat(folders["tclw"])
cp_ds = load_and_concat(folders["cp"])
lsp_ds = load_and_concat(folders["lsp"])

# ============================================
# 4️⃣ Process datasets in DataFrames
# ============================================
df_tcc = process_tcc(tcc_ds)
df_tclw = process_tclw(tclw_ds)
df_cp = process_precip(cp_ds, "cp", "convective_precip_mm")
df_lsp = process_precip(lsp_ds, "lsp", "large_scale_precip_mm")

dfs = [df_tcc, df_tclw, df_cp, df_lsp]
names = ["tcc", "tclw", "cp", "lsp"]

# ============================================
# 5️⃣ Cleaning, optimization, and combination
# ============================================
print("\n🔗 Cleaning and combining DataFrames...\n")

for i, (name, df) in enumerate(zip(names, dfs)):
    print(f"🔍 {name} shape before: {df.shape}")

    # Remove duplicate columns
    df = df.loc[:, ~df.columns.duplicated()]

    # Remove unnecessary columns
    for col in ["number", "expver", "valid_time"]:
        if col in df.columns:
            df = df.drop(columns=[col])

    # Convert to datetime
    df = df.copy()
    df["time"] = pd.to_datetime(df["time"], errors="coerce")

    # Convert floats to float32
    for col in df.select_dtypes(include=["float64"]).columns:
        df[col] = df[col].astype("float32")

    dfs[i] = df
    print(f"✅ {name} shape after: {df.shape}\n")

gc.collect()

# ============================================
# 6️⃣ Merge (outer join)
# ============================================
print("🔗 Combining all the DataFrames...\n")

df_final = dfs[0]
for name, next_df in zip(names[1:], dfs[1:]):
    print(f"➡️ Merge with {name} ({df_final.shape[0]} current rows)")
    df_final = pd.merge(df_final, next_df, on=["time", "latitude", "longitude"], how="outer", suffixes=("", "_drop"))
    # Remove possible duplicates of columns with the _drop suffix
    df_final = df_final.loc[:, ~df_final.columns.str.endswith("_drop")]
    gc.collect()

print("\n✅ Merge completed.")
print("📏 Final shape:", df_final.shape)
print("🧠 memory usage (MB):", df_final.memory_usage(deep=True).sum() / 1e6)




=== Procesando ERA5 variables atmosféricas ===
Cargando clouds:


100%|██████████| 5/5 [00:03<00:00,  1.59it/s]


✅ /content/drive/MyDrive/Colab Notebooks/clouds: combinado con éxito (5 archivos)
Cargando cloud_water:


100%|██████████| 5/5 [00:00<00:00, 23.30it/s]


✅ /content/drive/MyDrive/Colab Notebooks/cloud_water: combinado con éxito (5 archivos)
Cargando conv_prep:


100%|██████████| 5/5 [00:00<00:00, 21.65it/s]


✅ /content/drive/MyDrive/Colab Notebooks/conv_prep: combinado con éxito (5 archivos)
Cargando large_prep:


100%|██████████| 5/5 [00:00<00:00, 20.50it/s]


✅ /content/drive/MyDrive/Colab Notebooks/large_prep: combinado con éxito (5 archivos)

🔗 Limpiando y combinando DataFrames...

🔍 tcc shape antes: (34344, 6)
✅ tcc shape después: (34344, 4)

🔍 tclw shape antes: (34344, 6)
✅ tclw shape después: (34344, 4)

🔍 cp shape antes: (34344, 6)
✅ cp shape después: (34344, 4)

🔍 lsp shape antes: (34344, 6)
✅ lsp shape después: (34344, 4)

🔗 Combinando todos los DataFrames...

➡️ Merge con tclw (34344 filas actuales)
➡️ Merge con cp (34344 filas actuales)
➡️ Merge con lsp (34344 filas actuales)

✅ Merge completo.
📏 Shape final: (34344, 7)
🧠 Uso de memoria (MB): 1.09914


In [ ]:
df_final.head()

,time,latitude,longitude,cloud_cover,cloud_liquid_water,convective_precip_mm,large_scale_precip_mm
0,2021-07-01 00:00:00,39.650002,-75.760002,79.927299,0.362266,0.041993,0.000858
1,2021-07-01 01:00:00,39.650002,-75.760002,70.520851,0.326099,0.055280,0.000000
2,2021-07-01 02:00:00,39.650002,-75.760002,47.765263,0.186893,0.000000,0.000000
3,2021-07-01 03:00:00,39.650002,-75.760002,49.886131,0.159168,0.000000,0.000000
4,2021-07-01 04:00:00,39.650002,-75.760002,39.890034,0.152084,0.000000,0.000000


In [ ]:
import pandas as pd
import numpy as np

# --- Check structure ---

# --- Step 1: Ensure correct temporal order and types ---
df_final["time"] = pd.to_datetime(df_final["time"], errors="coerce")
df_final = df_final.dropna(subset=["time"])
df_final = df_final.sort_values("time")

# --- Step 2: Re-index by time---
df_final = df_final.set_index("time")

# --- Step 3: Create a new index every 15 minutes ---
time_15min = pd.date_range(start=df_final.index.min(),
                           end=df_final.index.max(),
                           freq="15min")

# --- Step 4: Reindex and interpolate ---
df_interp = df_final.reindex(time_15min)

# Continuous variables → linear interpolation
for col in ["cloud_cover", "cloud_liquid_water"]:
    df_interp[col] = df_interp[col].interpolate(method="time")

# Cumulative variables (precipitation) → maintain proportion
for col in ["convective_precip_mm", "large_scale_precip_mm"]:
    # Convert to rate (mm/hour → mm/15min)
    df_interp[col] = df_interp[col].interpolate(method="time") / 4.0

# --- Step 5: Reset index ---
df_interp = df_interp.reset_index().rename(columns={"index": "time"})

# --- Step 6: Final verification ---
print("\nAfter interpolating:")
print(df_interp.head(8))
print(f"\noriginal number of rows: {len(df_final)}")
print(f"Number of interpolated rows: {len(df_interp)}")

output_path = f"{base_path}/era5_atmospheric_combined_2021_2025.csv"
df_interp.to_csv(output_path, index=False)

print(f"\n✅ File exported successfully: {output_path}")


Después de interpolar:
                 time   latitude  longitude  cloud_cover  cloud_liquid_water  \
0 2021-07-01 00:00:00  39.650002 -75.760002    79.927299            0.362266   
1 2021-07-01 00:15:00        NaN        NaN    77.575684            0.353224   
2 2021-07-01 00:30:00        NaN        NaN    75.224075            0.344183   
3 2021-07-01 00:45:00        NaN        NaN    72.872467            0.335141   
4 2021-07-01 01:00:00  39.650002 -75.760002    70.520851            0.326099   
5 2021-07-01 01:15:00        NaN        NaN    64.831955            0.291298   
6 2021-07-01 01:30:00        NaN        NaN    59.143059            0.256496   
7 2021-07-01 01:45:00        NaN        NaN    53.454159            0.221695   

   convective_precip_mm  large_scale_precip_mm  
0              0.010498               0.000215  
1              0.011329               0.000161  
2              0.012159               0.000107  
3              0.012990               0.000054  
4         

In [ ]:
print("📄 Dimensiones:", df.shape)
print(df.head())


print("\n🔍 Valores nulos por columna:")
print(df.isna().sum())


print("\n🌡️ Temperatura [°C]:", df["temperature_C"].min(), "→", df["temperature_C"].max())
print("💧 Humedad relativa [%]:", df["relative_humidity"].min(), "→", df["relative_humidity"].max())
print("☁️ Nubosidad [%]:", df["cloud_cover"].min(), "→", df["cloud_cover"].max())
print("🌧️ Precipitación [mm/h]:", df["precipitation_mm"].min(), "→", df["precipitation_mm"].max())

📄 Dimensiones: (137373, 10)
                 time  latitude  longitude  temperature_C  dewpoint_C  \
0 2021-07-01 00:00:00     39.65     -75.76      29.683014   23.872711   
1 2021-07-01 00:15:00     39.65     -75.76      29.390045   23.858734   
2 2021-07-01 00:30:00     39.65     -75.76      29.097076   23.844757   
3 2021-07-01 00:45:00     39.65     -75.76      28.804108   23.830780   
4 2021-07-01 01:00:00     39.65     -75.76      28.511139   23.816803   

   relative_humidity  precipitation_mm  number  cloud_cover expver  
0          71.015945          0.286747       0    79.927301   0001  
1          72.198597          0.229043       0    77.575688   0001  
2          73.381248          0.171340       0    75.224075   0001  
3          74.563900          0.113636       0    72.872461   0001  
4          75.746552          0.055932       0    70.520848   0001  

🔍 Valores nulos por columna:
time                     0
latitude                 0
longitude                0
temperat